In [2]:
from pathlib import Path

from katabatic.artifacts import LocalArtifactStore
from katabatic.models.tabddpm.models import Tabddpm
from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from katabatic.utils.preprocess import preprocess_dataset


ROOT = None

for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / "datasets").exists() and (p / "models").exists():
        ROOT = p
        break


raw_file = ROOT / "datasets" / "nursery.csv"

processed_file = ROOT / "preprocessed_data" / "nursery_tabddpm.csv"

artifact_dir = ROOT / "artifacts"


print("ROOT:", ROOT)
print("Raw dataset:", raw_file)
print("Dataset exists:", raw_file.exists())


processed_file.parent.mkdir(
    parents=True,
    exist_ok=True
)


preprocess_dataset(
    str(raw_file),
    str(processed_file),
    target_col="8"
)


store = LocalArtifactStore(
    str(artifact_dir)
)


pipeline = TrainTestSplitPipeline(
    model=Tabddpm()
)

pipeline._evaluations = []


results = pipeline.run(
    input_csv=str(processed_file),
    dataset_name="nursery_tabddpm",
    artifact_store=store,
    model_name="tabddpm",
)


print(results)

ROOT: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic
Raw dataset: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic\datasets\nursery.csv
Dataset exists: True
Preprocessing: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic\datasets\nursery.csv
Saved preprocessed dataset to: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic\preprocessed_data\nursery_tabddpm.csv
Loaded data with shape: (12960, 9)
Train label distribution:
 8
not_recom     0.333333
priority      0.329186
spec_prior    0.312018
very_recom    0.025270
recommend     0.000193
Name: proportion, dtype: float64
Test label distribution:
 8
not_recom     0.333333
priority      0.329090
spec_prior    0.312114
very_recom    0.025463
Name: proportion, dtype: float64
Saved dataset artifact under datasets/nursery_tabddpm/split-20260831-052530
Step 100/200 | MLoss: 1.2578 | GLoss: 0.0000
Step 200/200 | MLoss: 1.

In [4]:
import pandas as pd

from katabatic.pipeline.evaluation_pipeline import SyntheticEvaluationPipeline


splits_root = ROOT / "artifacts" / "datasets" / "nursery_tabddpm"

split_dirs = sorted(
    [p for p in splits_root.glob("split-*") if p.is_dir()],
    key=lambda p: p.stat().st_mtime
)

latest_split = split_dirs[-1]

print("Using split:", latest_split)


train_df = pd.read_csv(
    latest_split / "train" / "train_full.csv"
)

test_df = pd.read_csv(
    latest_split / "test" / "test_full.csv"
)


target_col = "8"

categorical_cols = [
    "0",
    "1",
    "2",
    "3",
    "4",
    "5",
    "6",
    "7",
]

continuous_cols = []


model = pipeline.model

synthetic_df = model.sample(
    len(train_df),
    seed=42
)


print("Synthetic type:", type(synthetic_df))

print("Real columns:", train_df.columns.tolist())
print("Synthetic columns:", synthetic_df.columns.tolist())

print("\nSynthetic sample:")
print(synthetic_df.head())


evaluation_pipeline = SyntheticEvaluationPipeline(
    dimensions=[
        "fidelity",
        "utility",
        "diversity",
        "privacy",
        "consistency",
        "stability",
    ],
    categorical_cols=categorical_cols,
    continuous_cols=continuous_cols,
)


report = evaluation_pipeline.run(
    real_data=train_df,
    synthetic_data=synthetic_df,
    target_col=target_col,
    test_data=test_df,
    model=model,
)


print("\nComposite Score:", report.composite_score)

print("\nDimension Scores:")

for dimension, score in report.dimension_scores.items():
    print(f"{dimension}: {score}")

Using split: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic\artifacts\datasets\nursery_tabddpm\split-20260831-052530
Synthetic type: <class 'pandas.core.frame.DataFrame'>
Real columns: ['0', '1', '2', '3', '4', '5', '6', '7', '8']
Synthetic columns: ['0', '1', '2', '3', '4', '5', '6', '7', '8']

Synthetic sample:
            0         1         2  3           4           5        6  \
0  great_pret  critical  complete  1  convenient  convenient  nonprob   
1  great_pret  critical  complete  1  convenient  convenient  nonprob   
2  great_pret  critical  complete  1  convenient  convenient  nonprob   
3  great_pret  critical  complete  1  convenient  convenient  nonprob   
4  great_pret  critical  complete  1  convenient  convenient  nonprob   

           7          8  
0  not_recom  not_recom  
1  not_recom  not_recom  
2  not_recom  not_recom  
3  not_recom  not_recom  
4  not_recom   priority  

Running fidelity evaluation...

=== Fidelity Evaluation

c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\.venv\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\.venv\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\.venv\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\.venv\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\savin\O


=== Utility Evaluation ===
Overall utility score: 0.3557

Classifier   Metric     TSTR mean    TRTR mean    Delta   
--------------------------------------------------------
LR           accuracy   0.3333       0.7617       0.4284
LR           f1         0.1667       0.7545       0.5878
DT           accuracy   0.3333       0.9904       0.6571
DT           f1         0.1667       0.9906       0.8239
RF           accuracy   0.3333       0.9766       0.6433
RF           f1         0.1667       0.9762       0.8095
LinearSVM    accuracy   0.3333       0.7505       0.4172
LinearSVM    f1         0.1667       0.7423       0.5756
MLP          accuracy   0.3316       0.9984       0.6668
MLP          f1         0.1652       0.9986       0.8334

Running diversity evaluation...

=== Diversity Evaluation ===
Overall diversity score: 0.3166

Category Coverage (% of real categories in synth)
  0                              33.3%
  1                              20.0%
  2                            